# LLM Comparison: multi-model extraction benchmark

Side-by-side extraction quality and speed comparison using the 4-agent pipeline on 5 receipts.

| Model | Status |
|---|---|
| `qwen2.5:7b-instruct-q4_K_M` | baseline |
| `qwen2.5:14b-instruct` | challenger |
| `llama3.1` | challenger |
| `mistral:7b` | challenger |

Models not yet pulled in Ollama are automatically skipped with a pull command hint.

## Setup

In [1]:
import sys
from pathlib import Path

# Make sure the local src is on the path when running from the notebooks/ directory
repo_root = Path().resolve().parent  # finamt/notebooks -> finamt/
src_path = repo_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

import time
import warnings
import pandas as pd

warnings.filterwarnings("ignore")

from finamt import FinanceAgent
from finamt.agents.config import AgentsConfig

print("finamt imported successfully")

finamt imported successfully


In [2]:
# ── Receipt files ────────────────────────────────────────────────────────────
DATA_DIR = repo_root / "data"
receipts = sorted(DATA_DIR.glob("*.pdf"))
assert len(receipts) == 5, f"Expected 5 PDFs, found {len(receipts)}"
print("Receipts to process:")
for r in receipts:
    print(f"  {r.name}")

# ── Models to compare ────────────────────────────────────────────────────────
MODELS = {
    "qwen2.5:7b-instruct-q4_K_M": AgentsConfig(agent_model="qwen2.5:7b-instruct-q4_K_M"),
    "qwen2.5:14b-instruct":        AgentsConfig(agent_model="qwen2.5:14b-instruct"),
    "llama3.1:latest":                    AgentsConfig(agent_model="llama3.2:latest"),
    "mistral:7b":                    AgentsConfig(agent_model="mistral:7b"),
}


Receipts to process:
  adobe-2601.pdf
  anthropic-2601.pdf
  github-2601.pdf
  microsoft-2601.pdf
  openai-2601.pdf


## Run Extraction

Each model processes all 5 receipts sequentially using the 4-agent pipeline.  
Persistence is disabled (`db_path=None`) so no data is written to disk.

In [30]:
import requests

def _check_ollama_models(base_url: str = "http://localhost:11434") -> set[str]:
    """Return set of model names currently available in Ollama."""
    try:
        resp = requests.get(f"{base_url}/api/tags", timeout=5)
        resp.raise_for_status()
        return {m["name"] for m in resp.json().get("models", [])}
    except Exception as e:
        print(f"[warn] Could not reach Ollama: {e}")
        return set()

available = _check_ollama_models()
print("Models available in Ollama:")
for m in sorted(available):
    print(f"  {m}")

print()
missing = [name for name in MODELS if name not in available]
if missing:
    print("⚠  NOT available (will be skipped):")
    for m in missing:
        print(f"  {m}  →  run: ollama pull {m}")
else:
    print("✓ All configured models are available.")


Models available in Ollama:
  llama3.1:latest
  llama3.2:latest
  mistral:7b
  qwen2.5:14b-instruct
  qwen2.5:7b-instruct-q4_K_M
  qwen3-vl:8b
  qwen3.5:4b
  qwen3:30b
  qwen3:8b

✓ All configured models are available.


In [31]:
def _fmt_result(result) -> str:
    """Compact single-line summary of extraction result for table display."""
    if not result.success:
        return f"ERROR: {result.error_message}"
    d = result.data
    cp = d.counterparty.name if d.counterparty else None
    date = d.receipt_date.date().isoformat() if d.receipt_date else None
    total = str(d.total_amount) if d.total_amount is not None else None
    items = len(d.items) if d.items else 0
    cat = str(d.category) if d.category else None
    # Detect empty extraction (model not available / returned nothing)
    if not cp and not date and not total:
        return "⚠ EMPTY — model likely not available or returned no data"
    return f"{cp or '—'} | {date or '—'} | {(total or '—') + ' EUR'} | {items} items | {cat or '—'}"


# ── Main extraction loop ──────────────────────────────────────────────────────
raw_results: dict[str, list[dict]] = {}   # model_name -> list of row dicts

for model_name, agents_cfg in MODELS.items():
    if model_name not in available:
        print(f"\n⚠  Skipping {model_name} — not available in Ollama (run: ollama pull {model_name})")
        raw_results[model_name] = [
            {"receipt": pdf.name, "success": False, "summary": f"SKIPPED — run: ollama pull {model_name}", "time_s": 0.0}
            for pdf in receipts
        ]
        continue

    print(f"\n{'─'*60}")
    print(f"Model: {model_name}")
    print(f"{'─'*60}")
    agent = FinanceAgent(agents_cfg=agents_cfg, db_path=None)
    rows = []
    for pdf in receipts:
        t0 = time.perf_counter()
        result = agent.process_receipt(pdf)
        elapsed = round(time.perf_counter() - t0, 2)
        rows.append({
            "receipt":  pdf.name,
            "success":  result.success,
            "summary":  _fmt_result(result),
            "time_s":   elapsed,
        })
        status = "✓" if result.success else "✗"
        print(f"  {status}  {pdf.name:<35}  {elapsed:>6.2f}s")
    raw_results[model_name] = rows

print("\nDone.")


────────────────────────────────────────────────────────────
Model: qwen2.5:7b-instruct-q4_K_M
────────────────────────────────────────────────────────────
[finamt] adobe-2601.pdf
  [22:14:25] → PyMuPDF ...
  [22:14:25] → PDF text layer (page 1)
  [22:14:25] → Extraction pipeline ...
  [22:14:25] → Agent 1: metadata
  [22:14:35] → Agent 2: counterparty
  [22:14:41] → Agent 3: amounts
  [22:14:45] → Agent 4: line items
  ✓  adobe-2601.pdf                        23.98s
[finamt] anthropic-2601.pdf
  [22:14:49] → PyMuPDF ...
  [22:14:49] → PDF text layer (page 1)
  [22:14:49] → Extraction pipeline ...
  [22:14:49] → Agent 1: metadata
  [22:14:52] → Agent 2: counterparty
  [22:14:57] → Agent 3: amounts
  [22:15:00] → Agent 4: line items
  ✓  anthropic-2601.pdf                    14.01s
[finamt] github-2601.pdf
  [22:15:03] → PyMuPDF ...
  [22:15:03] → PDF text layer (page 1)
  [22:15:03] → Extraction pipeline ...
  [22:15:03] → Agent 1: metadata
  [22:15:06] → Agent 2: counterparty
  [22:1

## Results — Side-by-Side Comparison

In [32]:
model_names = list(MODELS.keys())

cols = {}
for m in model_names:
    df = pd.DataFrame(raw_results[m]).set_index("receipt")
    cols[f"{m} — extraction"] = df["summary"]
    cols[f"{m} — time (s)"]   = df["time_s"]

comparison = pd.DataFrame(cols)

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 400)
comparison


,qwen2.5:7b-instruct-q4_K_M — extraction,qwen2.5:7b-instruct-q4_K_M — time (s),qwen2.5:14b-instruct — extraction,qwen2.5:14b-instruct — time (s),llama3.1:latest — extraction,llama3.1:latest — time (s),mistral:7b — extraction,mistral:7b — time (s)
receipt,,,,,,,,
adobe-2601.pdf,Adobe Systems Software Ireland Ltd | 2026-01-30 | 55.84 EUR | 1 items | software,23.98,Adobe Systems Software Ireland Ltd | 2026-01-30 | 55.84 EUR | 1 items | software,47.87,Adobe Systems Software Ireland Ltd | 2026-01-30 | 65183.14 EUR | 1 items | software,9.02,Adobe Systems Software Ireland Ltd | 2026-01-30 | 55.84 EUR | 1 items | software,23.51
anthropic-2601.pdf,"Anthropic, PBC | 2026-01-21 | 6.0 EUR | 1 items | other",14.01,"Anthropic, PBC | 2026-01-21 | 6.0 EUR | 1 items | financial",29.13,"Anthropic, PBC | 2026-01-21 | 6.0 EUR | 1 items | services",4.99,"Anthropic, PBC | 2026-01-21 | 6.0 EUR | 1 items | other",13.25
github-2601.pdf,"GitHub, Inc. | 2026-01-11 | 10.0 EUR | 1 items | services",14.48,"GitHub, Inc. | 2026-01-11 | 10.0 EUR | 1 items | software",29.22,"GitHub, Inc. | 2026-01-11 | — EUR | 1 items | services",5.42,"GitHub, Inc. | 2026-01-11 | 10.0 EUR | 1 items | software",14.33
microsoft-2601.pdf,Microsoft Ireland Operations Ltd | 2026-01-02 | 13.92 EUR | 1 items | services,24.07,Microsoft Ireland Operations Ltd | 2026-01-02 | 13.92 EUR | 1 items | services,51.85,Microsoft Ireland Operations Ltd | 2026-01-02 | 13.92 EUR | 0 items | services,8.78,Microsoft Ireland Operations Ltd | 2026-01-02 | 13.92 EUR | 1 items | services,26.60
openai-2601.pdf,"OpenAI, LLC | 2026-01-27 | 10.0 EUR | 1 items | services",13.78,"OpenAI, LLC | 2026-01-27 | 10.0 EUR | 1 items | software",29.31,"OpenAI, LLC | 2026-01-27 | — EUR | 1 items | services",5.31,"OpenAI, LLC | 2026-01-27 | 10.0 EUR | 1 items | software",14.06


## Timing Summary

In [33]:
timing = pd.DataFrame({
    "model":       model_names,
    "total_s":     [sum(r["time_s"] for r in raw_results[m]) for m in model_names],
    "avg_s":       [sum(r["time_s"] for r in raw_results[m]) / len(receipts) for m in model_names],
    "successes":   [sum(r["success"] for r in raw_results[m]) for m in model_names],
}).set_index("model")

timing["total_s"] = timing["total_s"].round(2)
timing["avg_s"]   = timing["avg_s"].round(2)

timing

,total_s,avg_s,successes
model,,,
qwen2.5:7b-instruct-q4_K_M,90.32,18.06,5
qwen2.5:14b-instruct,187.38,37.48,5
llama3.1:latest,33.52,6.70,5
mistral:7b,91.75,18.35,5
